# ETL Silver - Tabla de Posiciones Champions League

## Objetivo
Limpiar y enriquecer los datos crudos de posiciones desde la capa **bronze** aplicando:

1. **Enriquecimiento de grupos**: Obtener información de grupos desde la tabla de partidos
2. **Filtrado de duplicados**: Eliminar tablas intermedias y vistas HOME/AWAY
3. **Recálculo de posiciones**: Calcular posiciones correctas dentro de cada grupo
4. **Estandarización de tipos**: Convertir campos a tipos de datos apropiados

---

## Fuentes
- **Bronze**: `proyecto_champions_league.bronze.posiciones_bronze` (datos crudos de la API)
- **Bronze**: `proyecto_champions_league.bronze.partidos_bronze` (para obtener grupos)

## Destino
- **Silver**: `proyecto_champions_league.silver.posiciones_silver`

In [0]:
%sql
-- Extraer la información de grupos desde los partidos
-- La API /standings NO incluye grupos, pero /matches SÍ

CREATE OR REPLACE TEMP VIEW equipos_grupos AS
SELECT DISTINCT
    temporada,
    local_id as equipo_id,
    grupo
FROM proyecto_champions_league.bronze.partidos_bronze
WHERE stage = 'GROUP_STAGE'
  AND grupo IS NOT NULL
  AND grupo != 'None'

UNION

SELECT DISTINCT
    temporada,
    visitante_id as equipo_id,
    grupo
FROM proyecto_champions_league.bronze.partidos_bronze
WHERE stage = 'GROUP_STAGE'
  AND grupo IS NOT NULL
  AND grupo != 'None'

In [0]:
%sql
-- Enriquecer posiciones con grupos y quedarnos solo con la tabla final
-- (máximo de partidos jugados por temporada/grupo)

CREATE OR REPLACE TEMP VIEW posiciones_con_grupos AS
SELECT 
    p.temporada,
    COALESCE(eg.grupo, p.grupo) as grupo,
    p.posicion,
    p.equipo_id,
    p.equipo_nombre,
    CAST(p.jugados AS INT) as jugados,
    CAST(p.ganados AS INT) as ganados,
    CAST(p.empatados AS INT) as empatados,
    CAST(p.perdidos AS INT) as perdidos,
    CAST(p.goles_favor AS INT) as goles_favor,
    CAST(p.goles_contra AS INT) as goles_contra,
    CAST(p.diferencia AS INT) as diferencia,
    CAST(p.puntos AS INT) as puntos
FROM proyecto_champions_league.bronze.posiciones_bronze p
LEFT JOIN equipos_grupos eg
    ON p.temporada = eg.temporada
    AND p.equipo_id = eg.equipo_id

In [0]:
%sql
select
*
from posiciones_con_grupos
where temporada = 2023 and equipo_id = 65

In [0]:
%sql
-- Eliminar tablas intermedias y vistas HOME/AWAY
-- Quedarnos solo con la tabla FINAL (máximo de partidos jugados)

CREATE OR REPLACE TEMP VIEW posiciones_finales AS
WITH max_partidos_por_temporada AS (
    SELECT 
        temporada,
        grupo,
        MAX(jugados) as max_jugados
    FROM posiciones_con_grupos
    GROUP BY temporada, grupo
)
SELECT 
    p.*
FROM posiciones_con_grupos p
INNER JOIN max_partidos_por_temporada m
    ON p.temporada = m.temporada
    AND p.grupo = m.grupo
    AND p.jugados = m.max_jugados

In [0]:
%sql
-- El campo 'posicion' de la API es una posición global
-- Recalcular posiciones correctas dentro de cada grupo/temporada

INSERT INTO proyecto_champions_league.silver.posiciones_silver (
    id, temporada, grupo, posicion, equipo_id,
    jugados, ganados, empatados, perdidos,
    goles_favor, goles_contra, diferencia, puntos
)
SELECT 
    MD5(CONCAT_WS('|', temporada, grupo, posicion)) id,
    CAST(temporada AS INT) temporada,
    grupo,
    -- Recalcular posición dentro del grupo ordenando por puntos
    ROW_NUMBER() OVER (
        PARTITION BY temporada, grupo 
        ORDER BY puntos DESC, 
                 diferencia DESC,
                 goles_favor DESC
    ) as posicion,
    CAST( equipo_id AS INT) equipo_id,
    jugados,
    ganados,
    empatados,
    perdidos,
    goles_favor,
    goles_contra,
    diferencia,
    puntos
FROM posiciones_finales